# Genomic Data Exploration

In this notebook, we will explore the **ClinVar** dataset to understand the structure of genomic variants and their clinical significance.

## 1. Setup

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.utils.vcf_parser import SimpleVCFParser
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

## 2. Load Data
We use our custom `SimpleVCFParser` to read the raw VCF file.

In [ ]:
vcf_path = "../data/raw/clinvar.vcf.gz"
parser = SimpleVCFParser(vcf_path)
parser.parse_header()

print(f"Header lines: {len(parser.header)}")
print("Sample names:", parser.samples)

## 3. Visualize Variant Types
Let's load the first 10,000 variants to get a quick overview.

In [ ]:
variants = []
count = 0
limit = 10000

for v in parser.parse_variants():
    variants.append(v)
    count += 1
    if count >= limit:
        break
        
df = pd.DataFrame(variants)
print(f"Loaded {len(df)} variants")
df.head()

In [ ]:
# Extract Length of Reference vs Alternate Allele
df['REF_len'] = df['REF'].apply(len)
df['ALT_len'] = df['ALT'].apply(lambda x: len(x[0]))

def classify_variant(row):
    if row['REF_len'] == 1 and row['ALT_len'] == 1:
        return 'SNP' # Single Nucleotide Polymorphism
    else:
        return 'Indel' # Insertion/Deletion

df['Type'] = df.apply(classify_variant, axis=1)

# Plot
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Type')
plt.title('Distribution of Variant Types (First 10k)')
plt.show()